<div style="background:linear-gradient(135deg,#0a2540 0%,#1a3a5c 60%,#0f3460 100%);
            padding:40px 30px;border-radius:12px;text-align:center;margin-bottom:10px;">
  <h1 style="color:#f4a261;font-size:2em;margin:0 0 8px;">
    🔬 Ciencia de Datos en Descubrimiento de Fármacos
  </h1>
  <h2 style="color:#a8dadc;font-size:1.2em;font-weight:400;margin:0 0 16px;">
    11 · PyTorch: Dropout · BatchNorm
  </h2>
  <p style="color:#cdd6f4;font-size:0.95em;max-width:640px;margin:0 auto;line-height:1.6;">
    Universidad Nacional de Colombia · Extensión UNAL 2026<br>
    <em>Semana 5 — Del dato curado al modelo predictivo</em>
  </p>
</div>

# PyTorch

---

### En esta lección aprenderás:
* cómo programar una red neuronal simple usando PyTorch.
* cómo implementar el descenso de gradiente estocástico (SGD) con minibatches.
* qué son las capas de Dropout y BatchNorm.
* cómo funcionan los optimizadores Adam y SGD.


### Tensores

Mientras que hasta ahora has trabajado con arrays de `numpy`, hoy usaremos `tensores`, más precisamente tensores de PyTorch.

Los tensores son muy similares a los arrays de numpy — también son arrays multidimensionales. 
La diferencia más importante es que PyTorch puede calcular gradientes automáticamente para los tensores. 
Esto es esencial para la retropropagación en redes neuronales.

Otra diferencia es que los tensores pueden ejecutarse en una GPU, lo que acelera enormemente el entrenamiento.

Los cálculos con `tensores` son casi idénticos a los cálculos con `np.arrays`. Pero las funciones pueden tener nombres diferentes. 
Por ejemplo, `np.matmul` se convierte en `torch.matmul`. 
El resultado de la transformación lineal de la semana pasada también se puede calcular con tensores de PyTorch:

In [ ]:
import torch # loads PyTorch

In [ ]:
X = torch.tensor([[1,2,3],
                [4,5,6]])

W = torch.tensor([[8,9,10],
                 [11,12,313]])

b = torch.tensor([1,2])

torch.mm(X,W.t())+b

Esta es la transformación lineal que ya conoces.<br>
Sin embargo, PyTorch simplifica este paso.
En PyTorch la transformación lineal ya está implementada como una función. Sigue leyendo para aprender cómo.

# Red Neuronal con PyTorch

In [ ]:
from torch import nn

El submódulo `nn` proporciona, entre otras cosas, la función `nn.Linear`. Realiza la transformación lineal $xW^T + b$.
Al igual que la semana pasada, necesitas especificar el número de features de entrada (`in_features`) y el número de neuronas de salida (`out_features`).

¿Falta la entrada para las capas?

In [ ]:
layer_1 = nn.Linear(in_features = 784, out_features=300, bias=True)

¿Falta la entrada para las capas?

Exactamente — hasta ahora tampoco has realizado ninguna transformación lineal, solo has definido la capa. 
La transformación lineal solo se realiza cuando se pasa la entrada a la capa. Lo verás en breve.

Una característica práctica de las capas `nn` es que los pesos de estas capas son inicializados automáticamente por PyTorch. 
Esto también facilita mucho las cosas. Veamos los pesos de la primera capa:

In [ ]:
list(layer_1.parameters())[0]

In [ ]:
list(layer_1.parameters())[0].shape

Como puedes ver, la matriz de pesos tiene el mismo tamaño que la semana pasada. También puedes ver que la matriz actúa como un `tensor` de PyTorch.

In [ ]:
import numpy as np
def min_max(x):
    return (x - np.min(x)) / (np.max(x) - np.min(x))


train_data = np.genfromtxt('https://uni-muenster.sciebo.de/s/b7DBTefJhTHZoGx/download', delimiter=',', skip_header =False) #genfromtxt reads .txt files if we chose delimiter ="," the function can read also .csv files  (comma seperated values)

train_images = min_max(train_data[:,1:])
train_images = torch.tensor(train_images, dtype = torch.float32)
train_labels=torch.tensor(train_data[:,0].astype(int), dtype = torch.long) 


test_data = np.genfromtxt('https://uni-muenster.sciebo.de/s/snnGZLWHjECihI5/download', delimiter=',', skip_header =False) #genfromtxt reads .txt files if we chose delimiter ="," the function can read also .csv files  (comma seperated values)

test_images = min_max(test_data[:,1:])
test_images = torch.tensor(test_images, dtype = torch.float32)
test_labels=torch.tensor(test_data[:,0].astype(int), dtype = torch.long) 

train_images.shape

El dataset contiene 60.000 imágenes con 784 píxeles cada una.
Ahora puedes usar estas como entrada para la transformación lineal.

In [ ]:
z_1=layer_1(train_images)
print(z_1)
z_1.shape

El `layer_1` devuelve la salida (`z_1`). Esta tiene la forma `[60000,300]`. 
Es decir, siguen siendo 60.000 imágenes, pero esta vez cada imagen tiene 300 valores en lugar de 784. 
La transformación lineal comprimió la información de 784 píxeles en 300 valores.

In [ ]:
from torch.nn import functional as F

a_1 = F.relu(z_1)
print(a_1)

Si comparas `a_1` con `z_1`, puedes ver que todos los valores que eran negativos antes se convirtieron en cero 
y todos los valores que eran positivos permanecieron iguales. 
Así funciona la función de activación **ReLU** (Rectified Linear Unit):

$$\text{ReLU}(x) = \max(0, x)$$

In [ ]:
layer_2 = nn.Linear(300,10) # 10 is the number of out_features, since we have 10 digits.
z_2=layer_2(a_1)

Nuevamente necesitas una función de activación, pero esta vez la función `softmax` para obtener las probabilidades. 
`nn.functional` tiene muchas funciones de activación ya implementadas:

In [ ]:
y_hat = F.softmax(z_2,dim=1) # the dim parameter defines whether the softmax function is applied over columns or rows.
y_hat.shape

En PyTorch también puedes combinar diferentes capas. Con `nn.Sequential` puedes escribir la transformación lineal y la activación de forma compacta:

In [ ]:
net = nn.Sequential(nn.Linear(784,300), 
                         nn.ReLU(), 
                         nn.Linear(300,10))
net

Como puedes ver, se ha creado una red con una capa oculta. Lo que debes notar es que en lugar de `F.relu` se usa `nn.ReLU`. 
La razón es que `nn.Sequential` espera capas (`nn`), no funciones (`F`).

In [ ]:
output = net(train_images)
output.shape

Otro cambio es que ya no usas la última función de activación. PyTorch la elige automáticamente. 
La decisión de qué función de activación usar en la última capa depende de la función de pérdida que se use. 
Aprenderás más sobre esto en breve.

### Función de Pérdida

`nn` también puede ayudar con la función de pérdida. Las funciones de pérdida más comunes ya están incluidas en PyTorch. 
La función de pérdida más usada para clasificación multiclase es la **entropía cruzada** (`CrossEntropyLoss`).

Importante: `CrossEntropyLoss` ya incluye internamente la función softmax, 
por eso no la aplicamos manualmente en la última capa de la red.

In [ ]:
loss_function = nn.CrossEntropyLoss()

La función `loss_function` puede ahora calcular la pérdida aplicando automáticamente la función softmax. 
Para ello, necesita la salida de la red (`output`) y las etiquetas correctas (`train_labels`):

In [ ]:
loss = loss_function(output, train_labels)
loss

### Retropropagación

El último paso es realizar la retropropagación. Gracias a *autograd* esto es fácilmente posible con PyTorch. 
Con `loss.backward()` PyTorch calcula automáticamente todos los gradientes:

```python
loss.backward()  # calcula los gradientes
```

El optimizador actualiza luego los pesos basándose en los gradientes. 
En este caso, usamos el **descenso de gradiente estocástico** (SGD):

In [ ]:
loss.backward() # collects the gradients

In [ ]:
from torch import optim
update_weights=optim.SGD(net.parameters(), lr=0.01) 
# You define which parameters and with which learning rate these should be changed.

update_weights.step()  # step() updates the weights


Ahora tienes todo lo que necesitas para entrenar una red.

Puedes usar nuevamente un `for-loop` para automatizar el entrenamiento.

Notarás que el código necesario es mucho más compacto que la semana pasada — PyTorch se encarga de la retropropagación automáticamente.

In [ ]:
## Define network, loss function and update algorithm
net = nn.Sequential(nn.Linear(784,300), 
                    nn.ReLU(), 
                    nn.Linear(300,10))

loss_function = nn.CrossEntropyLoss()
update = optim.SGD(net.parameters(), lr=0.3)
EPOCHS = 50

## Trainings Loop
for i in range(EPOCHS):
    update.zero_grad()
    output = net(train_images) # forward propagation
    
    loss   = loss_function(output, train_labels)
    loss.backward()
    acc=((output.max(dim=1)[1]==train_labels).sum()/float(output.shape[0])).item()
    print(i, 
        "Training Loss: %.2f Training Accuracy: %.2f"
        % (loss.item(), acc)
    )
    
    update.step()

Puedes ver que puedes entrenar una red neuronal con mucho menos esfuerzo. 
También puedes agregar una segunda o tercera capa oculta simplemente añadiendo más líneas a `nn.Sequential`.

In [ ]:
## Define network, loss function and update algorithm
net = nn.Sequential(nn.Linear(784,300), 
                    nn.ReLU(), 
                    nn.Linear(300,300),# <----- EXTRA LAYER
                    nn.ReLU(), 
                    nn.Linear(300,10))
print(net)
loss_function = nn.CrossEntropyLoss()
update = optim.SGD(net.parameters(), lr=0.3)
EPOCHS = 50

## Trainings Loop
for i in range(EPOCHS):
    update.zero_grad()
    output = net(train_images) # forward propagation
    
    loss   = loss_function(output, train_labels)
    loss.backward()
    
    acc=((output.max(dim=1)[1]==train_labels).sum()/float(output.shape[0])).item()
    print(i,
        "Training Loss: %.2f Training Accuracy: %.2f"
        % (loss.item(), acc)
    )
    
    update.step()

Es posible que hayas notado que estamos usando el Descenso de Gradiente Estocástico como optimizador (para actualizar los pesos). 
Hasta ahora, siempre hemos pasado todas las imágenes por la red a la vez (batch completo). 
Sin embargo, el **Descenso de Gradiente Estocástico** procesa los datos en pequeños lotes llamados **minibatches**. 
Esto tiene varias ventajas:

- Es más eficiente computacionalmente (especialmente con GPU)
- Introduce ruido beneficioso que ayuda a escapar de mínimos locales
- Permite entrenar con datasets que no caben en memoria

Para usar minibatches en PyTorch, necesitamos el módulo `torch.utils.data`:

Para aprovechar el Descenso de Gradiente Estocástico, primero debemos dividir los datos en minibatches. 
Para esto también hay herramientas útiles en PyTorch:

In [ ]:
from torch.utils import data

En `torch.utils.data` hay dos funciones que necesitas:

* `data.TensorDataset(input, labels)` crea un dataset de PyTorch a partir de tensores
* `data.DataLoader(dataset, batch_size=32, shuffle=True)` crea un iterador que devuelve minibatches

El parámetro `batch_size` define cuántas imágenes hay en cada minibatch. 
El parámetro `shuffle=True` mezcla los datos antes de crear los minibatches.

In [ ]:
train_data = data.TensorDataset(train_images, train_labels) 
# input are our tensors which contain the images and the labels
loader = data.DataLoader(train_data, batch_size = 32)

In [ ]:
print(len(loader))

La variable `loader` ahora contiene 1875 minibatches, cada uno con 32 imágenes y sus 32 etiquetas. 
En la siguiente celda puedes ver cómo se ve un minibatch:

In [ ]:
list(loader)[0]

Para unir todo, necesitas un segundo `for-loop` que seleccione los minibatches uno por uno dentro del primer `for-loop`:

In [ ]:
## Define network, loss function and update algorithm
net = nn.Sequential(nn.Linear(784,300), 
                    nn.ReLU(), 
                    nn.Linear(300,300),
                    nn.ReLU(), 
                    nn.Linear(300,10))
loss_function = nn.CrossEntropyLoss()
update = optim.SGD(net.parameters(), lr=0.3)
EPOCHS = 2

## Trainings Loop
for i in range(EPOCHS):
    loss_list = [] # this list stores the loss of each minibatch
                   # with this we can calculate the average loss within the epoch
    for minibatch in loader: # loop through all minibatches
        images, labels = minibatch # minibatch is divided into images and labels
        
        update.zero_grad()
        output = net(images) # forward propagation
    
        loss   = loss_function(output, labels)
        loss.backward()
        loss_list.append(loss.item())
        update.step()
        
    output = net(train_images)
    acc=((output.max(dim=1)[1]==train_labels).sum()/float(output.shape[0])).item()
    print(
        "Training Loss: %.2f Training Accuracy: %.2f"
        % (np.mean(loss_list), acc)
    )
    

Después de solo dos épocas, la exactitud es mucho mayor que antes. 
Una sola época tarda mucho más en comparación con el entrenamiento 'normal', 
porque el optimizador actualiza los pesos 1875 veces por época en lugar de una sola vez.

# Capas Avanzadas

A continuación trataremos nuevas capas que se usan además de las capas lineales.

## Dropout

El **Dropout** es una técnica de regularización que ayuda a prevenir el sobreajuste. 
Durante el entrenamiento, aleatoriamente 'apaga' algunas neuronas con una probabilidad definida. 
Esto obliga a la red a aprender representaciones más robustas.

Por ejemplo, con `nn.Dropout(0.5)`, el 50% de las neuronas se desactivan aleatoriamente en cada forward pass durante el entrenamiento:

In [ ]:
torch.manual_seed(1235)

example_x = torch.tensor([[1.,2.,3.,4.,5.]] )
do = nn.Dropout(0.5)
do(example_x)

Tres de los valores fueron establecidos en `0`, pero los otros valores se duplicaron. **¿Por qué ocurrió esto?**

Esto es porque entrenamos con solo el 50% de las neuronas activas. 
Si no se compensara, la red vería señales más débiles durante el entrenamiento que durante la evaluación. 
Para compensar, los valores restantes se multiplican por $\frac{1}{1-p}$ (en este caso por 2).

Con `.eval()` el dropout se desactiva — para que durante la evaluación todas las neuronas estén activas:

In [ ]:
do.eval()
do(example_x)

En modo `.eval()` el dropout no se aplica.

## BatchNorm

Las capas de **Batch Normalization** son otra capa comúnmente usada en redes neuronales profundas. 
Normalizan la salida de una capa para que tenga media 0 y desviación estándar 1. 
Esto estabiliza y acelera el entrenamiento.

Se coloca generalmente **después** de la transformación lineal y **antes** de la función de activación.

**¿Puedes completar el código?**

In [ ]:
batch_x, batch_y = next(iter(loader)) # here the first minibatch is chosen

layer_one = nn.Sequential(nn.Linear(____,___),
                         nn.BatchNorm1d(_____),
                         nn.ReLU(),
                         nn.Dropout(____))
layer_one(batch_x)

<details>
    <summary><b>Solución:</b></summary>

```python
layer_one = nn.Sequential(nn.Linear(784, 300),
                           nn.BatchNorm1d(300),
                           nn.ReLU())
```
</details>

Ahora extiende la red completa de antes.
Importante: ni BatchNorm ni Dropout se usan después de la última capa lineal.

In [ ]:
net= nn.Sequential(nn.Linear(784,300), 
                   ______________,
                   ______________,
                   ______________,
                   nn.Linear(300,300),
                   ______________,
                   ______________,
                   ______________,
                   nn.Linear(300,10))

<details>
    <summary><b>Solución:</b></summary>

```python
net = nn.Sequential(nn.Linear(784, 300),
                    nn.BatchNorm1d(300),
                    nn.ReLU(),
                    nn.Dropout(0.5),
                    nn.Linear(300, 10))
```
</details>

# Optimizadores

Los optimizadores determinan cómo se actualizan con precisión los pesos de la red. 
Hasta ahora siempre has usado el `SGD` (Descenso de Gradiente Estocástico). 
Existen otros optimizadores que pueden ser más eficientes en ciertos casos.

El optimizador más popular actualmente es **Adam** (Adaptive Moment Estimation). 
Adapta la tasa de aprendizaje individualmente para cada parámetro — generalmente converge más rápido que SGD.

In [ ]:
loss_function = nn.CrossEntropyLoss()
update = optim.Adam(net.parameters(), lr=0.1)
EPOCHS = 10

## Training Loop
for i in range(EPOCHS):
    loss_list = [] # in this list we save the loss of each minibatch so we can
                   # calculate the average loss at the end of the epoch
    net.train()
    for minibatch in loader: # loop through all minibatches
        images, labels = minibatch # divide minibatches in labels and images
        
        update.zero_grad()
        output = net(images) # forward propagation
    
        loss   = loss_function(output, labels)
        loss.backward()
        loss_list.append(loss.item())
        update.step()
    net.eval()    
    output = net(train_images)
    acc=((output.max(dim=1)[1]==train_labels).sum()/float(output.shape[0])).item()
    
    print(i,
        "Training Loss: %.2f Training Accuracy: %.2f"
        % (np.mean(loss_list), acc)
    )

# Ejercicio Práctico

Para el ejercicio, entrenarás una red, pero esta vez usando los datos de toxicidad del notebook 5.

El dataset ya fue cargado y los fingerprints calculados. Ahora necesitas:
1. Convertir los fingerprints y la actividad a tensores
2. Dividir en conjuntos de entrenamiento y prueba
3. Crear un DataLoader
4. Definir la red con el tamaño correcto
5. Completar el loop de entrenamiento

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import sys
if 'google.colab' in sys.modules: # checks whether the notebook runs on collab
    !wget https://raw.githubusercontent.com/kochgroup/intro_pharma_ai/main/utils/utils.py
    !pip install rdkit==2022.3.4
    %run utils.py
else:
    %run ../utils/utils.py # loads pre-written functions

In [ ]:
data_tox = pd.read_csv("https://raw.githubusercontent.com/filipsPL/tox21_dataset/master/compounds/sr-mmp.tab", sep = "\t")
data_tox = data_tox.iloc[:,1:] # all columns except the first (index 0) are chosen
data_tox.columns = ["smiles", "activity"]
data_tox.head()

A continuación, calculas los fingerprints. Al igual que en el notebook 5, la función `get_fingerprints` está disponible para este propósito.

In [ ]:
fps = get_fingerprints(data_tox)
fps["activity"] = data_tox.activity
fps.head()

Antes de poder usarlos en PyTorch, necesitas convertir tanto los fingerprints como la `activity` a `tensores`. 
Ten en cuenta que los fingerprints deben ser `float32` y la actividad debe ser `long` (entero):

In [ ]:
fps = torch.tensor(___.values, dtype=torch.float32) #

In [ ]:
train, test=train_test_split(fps,test_size= 0.2 , train_size= 0.8, random_state=1234)


train_x = train[:,:-1]
train_y = train[:,-1]
test_x = test[:,:-1]
test_y = test[:,-1]

Ahora queremos usar minibatches nuevamente. Para esto aún debemos convertir nuestros datos de entrenamiento en un `DataLoader`. 
¿Por qué solo los datos de entrenamiento?

Porque para la evaluación no necesitamos minibatches — podemos pasar todos los datos de prueba a la vez.

In [ ]:
train_data=data.TensorDataset(______, _____) # input are our tensors, for the fingerprints and the activities
loader=data.DataLoader(train_data, batch_size = 32)
len(loader)

Ajusta la red para que la entrada y la salida tengan el tamaño correcto. 
Es decir, la longitud de los fingerprints y el número de clases (activo/inactivo = 2 clases).

In [ ]:
net= nn.Sequential(nn.Linear(), 
                   nn.BatchNorm1d(),
                   nn.ReLU(), 
                   nn.Dropout(),
                   nn.Linear(),
                   nn.BatchNorm1d(),
                   nn.ReLU(), 
                   nn.Dropout(),
                   nn.Linear())

loss_function = nn.BCEWithLogitsLoss()
update = ___________________, lr=0.1)    
EPOCHS = 10

Por último, completa el `for loop`.

`.squeeze` convierte el tensor de salida `(n,1)` a un tensor 1-dimensional de longitud `n`.

In [ ]:
for i in range(EPOCHS):
    loss_list = [] # in this list we save the loss of each minibatch 
    net.train()
    for minibatch in loader: # loop through all minibatches
        update.__________
        molecules, activity = minibatch # divide minibatches in labels and molecules
        output = net(____________) # forward propagation
        loss   = loss_function(output.squeeze(), ____________)
        loss._______
        loss_list.append(loss.item())
        update.________
    # here the accuracy for the testset is calculated
    net.eval()
    output = net(test_x)
    acc = torch.sum((output>0).squeeze().int() == test_y)/float(test_y.shape[0])
   
    print(
        "Training Loss: %.2f Test Accuracy: %.2f"
        % (np.mean(loss_list), acc.item())
    )